In [ ]:
import pathlib
import platform
import pickle

import haiku as hk
import jax
import chex
import numpy as np
import hvplot.xarray
import panel as pn
import xarray as xr
import cartopy.crs as ccrs
import grain.python as grain

from jax.sharding import PartitionSpec, NamedSharding

from graphcast import xarray_jax, xarray_tree, checkpoint, typed_graph
from graphcast.casting import Bfloat16Cast
from graphcast.data_utils import extract_inputs_targets_forcings
from graphcast.mesh_connectivity import get_connected_mesh_nodes, mask_mesh
from graphcast.mesh_graph import faces_to_edges, MeshGraph
from graphcast.model import TASK, TaskConfig, ModelConfig, GraphCast
from graphcast.normalization import InputsAndResiduals
from graphcast.mask import MaskedPredictor
from graphcast.ocean_mesh_utils import read_mesh
from graphcast.xarray_jax import unwrap_vars, unwrap_data
from graphcast.dataloader import ARCODataSource, AddLogDepthCoordinate

pn.extension()

# Stack traces tend to be deep, if filtering is enabled useful information might be lost
jax.config.update("jax_traceback_filtering", 'off')

In [ ]:
data_source_path = '/leonardo_scratch/large/userexternal/scampane/xcast/data/dataset/dataset_tres-1d_res-0p25_levels-10_arco'
model_config_path = '/leonardo_scratch/large/userexternal/scampane/xcast/data/model_config.ckpt'
task_config_path = '/leonardo_scratch/large/userexternal/scampane/xcast/data/task_config.ckpt'
normalization_artifacts_path = '/leonardo_scratch/large/userexternal/scampane/xcast/data/dataset/dataset_tres-1d_res-0p25_levels-10_normalization'
mesh_data_path = '/leonardo_scratch/large/userexternal/scampane/xcast/data/mesh_data.ckpt'
batch_size = 1

In [ ]:
data_source = ARCODataSource(path=data_source_path, timesteps=3)
sampler = grain.IndexSampler(num_records=len(data_source), num_epochs=1, shuffle=True, seed=0)
operations = [AddLogDepthCoordinate(), # The negative logarithm of depth is used as a weight in loss calculations
              grain.Batch(batch_size=batch_size, drop_remainder=True, batch_fn=lambda datasets: xr.concat(datasets, dim='batch'))]
dataloader = grain.DataLoader(data_source=data_source, sampler=sampler, operations=operations, worker_count=0)

# Load a single minibatch from dataloader and exit
for dataset in dataloader:
    break
dataset

In [ ]:
devices = jax.devices()
devices

In [ ]:
devices_mesh = jax.make_mesh((batch_size, len(devices) // batch_size), ('batch', 'graph'), devices=devices)
devices_mesh

In [ ]:
def put_dataset(dataset, replicate_along_batch=False):
    if replicate_along_batch:
        sharding = NamedSharding(devices_mesh, PartitionSpec())
    else:
        sharding = NamedSharding(devices_mesh, PartitionSpec('batch'))

    def _put_dataarray(data_array):
        return jax.tree_util.tree_map(lambda xs: jax.device_put(xs, sharding), data_array)
        
    return dataset.map(lambda da: _put_dataarray(da))

In [ ]:
dataset_jax = xarray_jax.to_jax(dataset)
# Notice: as a side-effect, boolean variables get casted to float32 (which is useful)
dataset_jax = dataset_jax.fillna(value=jax.numpy.float32(0.0))

inputs, targets, forcings = extract_inputs_targets_forcings(dataset=dataset_jax, **TASK, target_lead_times="1d")
# Replicate inputs along batch dimension
inputs, targets, forcings = map(lambda x: put_dataset(x, replicate_along_batch=False), [inputs, targets, forcings])

In [ ]:
with open(model_config_path, 'rb') as file:
    model_config = checkpoint.load(file, ModelConfig)
    
model_config

In [ ]:
with open(task_config_path, 'rb') as file:
    task_config = checkpoint.load(file, TaskConfig)

task_config

In [ ]:
normalization_artifacts = xr.open_datatree(normalization_artifacts_path, engine='zarr')
normalization_artifacts = normalization_artifacts.map_over_datasets(lambda ds: put_dataset(ds, replicate_along_batch=True))

mean_by_level = normalization_artifacts['/inputs/location'].dataset
mean_by_level = mean_by_level.fillna(0.0)

stddev_by_level = normalization_artifacts['/inputs/scale'].dataset
stddev_by_level = stddev_by_level.fillna(1.0)
stddev_by_level = stddev_by_level.clip(min=1e-18)

diffs_stddev_by_level = normalization_artifacts['/residuals/scale'].dataset
diffs_stddev_by_level = diffs_stddev_by_level.fillna(1.0)
diffs_stddev_by_level = diffs_stddev_by_level.clip(min=1e-18)

In [ ]:
@chex.dataclass(frozen=True, eq=True)
class MeshData:
    mesh_graph: MeshGraph
    boundary_nodes: np.array

with open(mesh_data_path, 'rb') as file:
    mesh_data = checkpoint.load(file, MeshData)

mesh_data

In [ ]:
# Deeper one-step predictor.
predictor = GraphCast(model_config, 
                      task_config, 
                      grid_lat=dataset['lat'].to_numpy(), 
                      grid_lon=dataset['lon'].to_numpy(), 
                      grid_mask=data_source.mask,
                      mesh_graph=mesh_data.mesh_graph,
                      boundary_nodes=mesh_data.boundary_nodes,
                      ensure_divisible_by=len(devices))

def put_graph(graph):
    
    def _put_leaf(xs):
        pspec = jax.sharding.PartitionSpec() if len(xs.shape) == 1 else jax.sharding.PartitionSpec('graph')
        return jax.device_put(xs, jax.sharding.NamedSharding(devices_mesh, pspec))
        
    return jax.tree_util.tree_map(_put_leaf, graph)

for attr in ['_grid2mesh_graph_structure', '_mesh_graph_structure', '_mesh2grid_graph_structure']:
    setattr(predictor, attr, put_graph(getattr(predictor, attr))) 

# Modify inputs/outputs to `graphcast.GraphCast` to handle conversion to from/to float32 to/from BFloat16.
predictor = Bfloat16Cast(predictor)

# Modify inputs/outputs to `casting.Bfloat16Cast` so the casting to/from BFloat16 happens after applying normalization to the inputs/targets.
predictor = InputsAndResiduals(
    predictor,
    diffs_stddev_by_level=diffs_stddev_by_level,
    mean_by_level=mean_by_level,
    stddev_by_level=stddev_by_level)

# Mask inputs/outputs replacing missing values with 0.0
predictor = MaskedPredictor(predictor, mask=dataset['glorys_mask'], value=0.0)

In [ ]:
@hk.transform
def run_forward(inputs, targets_template, forcings):
  return predictor(inputs, targets_template=targets_template, forcings=forcings)

run_forward_jit = jax.jit(jax.checkpoint(loss_fn.apply, policy=jax.checkpoint_policies.nothing_saveable))

In [ ]:
key = jax.random.key(0)
params = run_forward.init(rng=key, inputs=inputs, targets_template=targets, forcings=forcings)
params = jax.tree_util.tree_map(lambda xs: jax.device_put(xs, NamedSharding(devices_mesh, PartitionSpec())), params)

In [ ]:
@hk.without_apply_rng
@hk.transform
def loss_fn(inputs, targets, forcings):
  loss, diagnostics = predictor.loss(inputs, targets, forcings=forcings, levels_normalization_coord='log-depth')
  return xarray_tree.map_structure(
      lambda x: unwrap_data(x.mean(), require_jax=True),
      (loss, diagnostics))

loss_value_and_grad = jax.value_and_grad(run_forward_jit, has_aux=True)

In [ ]:
# Needs a lot more than 16GB of RAM!
loss, grads = loss_value_and_grad(params, inputs, targets, forcings);

In [ ]:
%timeit jax.block_until_ready(loss_value_and_grad(params, inputs, targets, forcings));

In [ ]:
loss